<a href="https://colab.research.google.com/github/piramid678-arch/FallGuard-AI/blob/master/JEV%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%85%E1%85%B5_%E1%84%85%E1%85%A6%E1%84%8B%E1%85%B5%E1%84%89%E1%85%B3_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ JEV의 원리 — 같은 두뇌, 다른 사용법

**Connect AI LAB 실습** | 준비물: 없음 (API 키 불필요, 완전 무료)

2026년 9월, JEV라는 인공지능이 나와 해외가 떠들썩합니다. 문장을 생성하지 않고 **결정을 확률로 직접 출력**하는 모델인데, 기존 LLM보다 수십 배 빠르고 수백 배 쌉니다.

이 노트북에서는 작은 로컬 모델 하나로 그 **원리**를 직접 재현합니다.
- **소설가 방식**: 두뇌에게 문장(JSON)을 생성시키고 → 파싱한다 (기존 LLM 방식)
- **정책망 방식**: 두뇌에게 생성을 시키지 않고, 선택지의 확률만 한 번에 읽는다 (JEV의 원리)

같은 두뇌로 두 방식을 경주시켜 보세요. 위에서부터 ▶ 버튼만 누르면 됩니다.
(런타임 → 런타임 유형 변경 → T4 GPU 선택하면 더 빠릅니다)


## 1. 설치와 두뇌 로드 (약 1~2분)


In [ ]:
%pip install -q transformers accelerate


In [ ]:
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # 5억 파라미터짜리 작은 무료 두뇌
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()
print("두뇌 준비 완료")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

두뇌 준비 완료


## 2. 상황과 질문

고객 문의 하나를 놓고, 자동화 시스템이 알아야 할 결정 3개를 묻습니다.


In [ ]:
state = "스트라이프 결제 연동이 3일째 실패하고 있어요. 매출이 계속 빠지고 있습니다. 제발 빨리 도와주세요."

질문들 = [
    ("어느 팀이 처리해야 하는가", {"A": "결제팀", "B": "기술팀", "C": "영업팀"}),
    ("긴급한가", {"A": "예", "B": "아니오"}),
    ("고객의 감정 상태는", {"A": "차분함", "B": "답답하지만 정중함", "C": "매우 분노"}),
]

def 프롬프트(q, options):
    보기 = " / ".join(f"{k}: {v}" for k, v in options.items())
    return (f"고객 문의: {state}\n질문: {q}\n보기: {보기}\n"
            f"가장 알맞은 보기의 알파벳 하나만 고르시오.\n답:")


## 3. 소설가 방식 — 문장을 생성시키고 파싱하기

두뇌에게 "JSON으로만 답해"라고 부탁합니다. 기존 LLM 자동화가 일하는 방식입니다.


In [ ]:
# ── 방식 1: 소설가 (자기회귀 생성 → JSON 파싱) ──────────────
def 소설가():
    msgs = [{"role": "user", "content":
        f"고객 문의: {state}\n다음 질문들에 JSON으로만 답하세요. 다른 말 금지.\n"
        + json.dumps({q: list(o.values()) for q, o in 질문들}, ensure_ascii=False)}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    t0 = time.time()
    out = model.generate(**enc, max_new_tokens=120, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    dt = time.time() - t0
    new = out[0][enc["input_ids"].shape[1]:]
    text = tok.decode(new, skip_special_tokens=True)
    try:
        parsed = json.loads(text[text.find("{"): text.rfind("}") + 1])
        상태 = "파싱 성공"
    except Exception:
        parsed, 상태 = text[:80], "⚠️ 파싱 실패 (자동화 파이프라인이 여기서 깨짐)"
    return dt, len(new), parsed, 상태

dt1, ntok, parsed, 상태 = 소설가()
print(f"[소설가 방식]  {dt1:.2f}초 · 생성 토큰 {ntok}개 · {상태}")
print(json.dumps(parsed, ensure_ascii=False)[:200])


[소설가 방식]  17.52초 · 생성 토큰 39개 · 파싱 성공
{"결제팀": "3일째 실패", "긴급한가": "아니오", "고객의 감정 상태는": "매우 분노"}


## 4. 정책망 방식 — 생성 없이 확률만 읽기 (JEV의 원리)

---



같은 두뇌에게 이번엔 아무것도 생성시키지 않습니다. 질문과 보기를 보여주고, **다음에 올 글자의 점수판(로짓)에서 보기들의 확률만 읽습니다.** 단 한 번의 계산입니다.


In [ ]:
# ── 방식 2: 정책망 (생성 없이, 선택지의 확률만 한 번에 읽기) ──
def 정책망():
    t0 = time.time()
    답 = {}
    for q, options in 질문들:
        ids = tok(프롬프트(q, options), return_tensors="pt").input_ids
        with torch.no_grad():
            logits = model(ids).logits[0, -1]          # 다음 토큰의 점수판
        opt_ids = {k: tok.encode(k, add_special_tokens=False)[0] for k in options}
        probs = torch.softmax(torch.tensor([logits[i] for i in opt_ids.values()]), 0)
        분포 = {options[k]: round(float(p), 2) for k, p in zip(opt_ids, probs)}
        선택 = max(분포, key=분포.get)
        답[q] = {"choice": 선택, "confidence": 분포[선택], "probabilities": 분포}
    return time.time() - t0, 답

dt2, 답 = 정책망()
print(f"[정책망 방식]  {dt2:.2f}초 · 생성 토큰 0개 · 형태 보장")
print(json.dumps(답, ensure_ascii=False, indent=1))
print(f"\n같은 두뇌, 같은 질문 — 정책망이 {dt1/dt2:.1f}배 빠름")


[정책망 방식]  11.61초 · 생성 토큰 0개 · 형태 보장
{
 "어느 팀이 처리해야 하는가": {
  "choice": "기술팀",
  "confidence": 0.66,
  "probabilities": {
   "결제팀": 0.25,
   "기술팀": 0.66,
   "영업팀": 0.09
  }
 },
 "긴급한가": {
  "choice": "예",
  "confidence": 0.61,
  "probabilities": {
   "예": 0.61,
   "아니오": 0.39
  }
 },
 "고객의 감정 상태는": {
  "choice": "답답하지만 정중함",
  "confidence": 0.79,
  "probabilities": {
   "차분함": 0.15,
   "답답하지만 정중함": 0.79,
   "매우 분노": 0.06
  }
 }
}

같은 두뇌, 같은 질문 — 정책망이 1.5배 빠름


## 5. 결과 읽는 법

제이의 실측(맥북, 같은 코드) 기준:
- 소설가: 1.94초, 39토큰 생성 — 파싱은 됐지만 **내용이 틀렸습니다** (긴급한 문의를 '아니오', 정중한 고객을 '매우 분노'로 판정)
- 정책망: 0.41초, 생성 0토큰 — **전부 정답 + 확률까지** (기술팀 67%, 긴급 예 59%, 답답하지만 정중함 80%)

같은 두뇌인데 왜 다를까요? 생성은 한 글자씩 쓰며 흔들리지만, 채점은 선택지 위에서만 이뤄지기 때문입니다. 출력의 **형태가 계약으로 보장**되니 파싱도, 기도도 필요 없습니다.

JEV는 이 원리를 전용 모델로 만들어 훈련까지 강화학습으로 한 것입니다. 보정된 결정으로부터의 강화학습 — 만든 사람은 ChatGPT를 훈련시킨 RLHF의 공동 개발자입니다.

### 실험해 보세요
1. `state`를 여러분 상황으로 바꿔 보세요 (고객 문의, 이메일, 리뷰…)
2. 질문과 보기를 추가해 보세요 — 정책망 방식은 질문이 늘어도 생성 토큰이 0입니다
3. 보기 순서를 바꿔도 확률이 안정적인지 확인해 보세요

### 더 공부하기
- 무료 커리큘럼: https://www.aicitybuilders.com/guide
- 막히면 질문: https://www.aicitybuilders.com/ask
- 10월 「피지컬 AI 강화학습」 강의: https://www.aicitybuilders.com/robot

— AI 멘토 제이 · Connect AI LAB


## 6. 나만의 질문지 만들기 — 미니 systemone

JEV API의 실제 구조를 그대로 재현합니다. **질문과 보기는 사람이 미리 설계**하고, 모델은 매번 상황(state)을 받아 보기 위에 확률을 나눠줄 뿐입니다.

문항 형식은 JEV와 같은 세 가지입니다.
- **choice**: 보기 중 하나 고르기 → 선택 + 확률분포 + 확신도
- **noul**: 참인가? → 0~1
- **score**: 기준표 위의 점수


In [ ]:
def 확률읽기(state, instructions, options):
    보기 = " / ".join(f"{k}: {v}" for k, v in options.items())
    p = f"상황: {state}\n질문: {instructions}\n보기: {보기}\n가장 알맞은 보기의 알파벳 하나만 고르시오.\n답:"
    ids = tok(p, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    oi = {k: tok.encode(k, add_special_tokens=False)[0] for k in options}
    pr = torch.softmax(torch.stack([logits[i] for i in oi.values()]), 0)
    return {k: round(float(v), 3) for k, v in zip(options, pr)}

def systemone(state, questions):
    answers = {}
    for name, q in questions.items():
        if q["type"] == "noul":
            pr = 확률읽기(state, q["instructions"], {"A": "예", "B": "아니오"})
            answers[name] = {"type": "noul", "noul": pr["A"]}
        elif q["type"] == "choice":
            keys = list("ABCDEFG")[: len(q["criteria"])]
            options = dict(zip(keys, q["criteria"].keys()))
            설명 = {k: q["criteria"][options[k]] for k in keys}
            pr = 확률읽기(state, q["instructions"], 설명)
            probs = {options[k]: v for k, v in pr.items()}
            best = max(probs, key=probs.get)
            answers[name] = {"type": "choice", "choice": best,
                             "confidence": probs[best], "probabilities": probs}
        elif q["type"] == "score":
            options = {str(i): c for i, c in enumerate(q["criteria"])}
            pr = 확률읽기(state, q["instructions"], options)
            answers[name] = {"type": "score",
                             "score": round(sum(int(k) * v for k, v in pr.items()), 2),
                             "probabilities": pr}
    return answers

print("미니 systemone 준비 완료")


미니 systemone 준비 완료


### 실전: 유튜브 댓글 자동 분류

질문지는 한 번만 설계하고, 어떤 댓글이 들어와도 같은 질문지로 채점합니다.


In [ ]:
질문지 = {
    "댓글유형": {"type": "choice", "instructions": "이 댓글의 유형은",
        "criteria": {"감사": "고마움 표현", "질문": "도움 요청이나 궁금증", "스팸": "홍보나 무관한 내용"}},
    "답변필요": {"type": "noul", "instructions": "채널 운영자의 실질적 답변이 필요한가"},
    "난이도": {"type": "score", "instructions": "답변에 필요한 기술 난이도",
        "criteria": ["인사만 하면 됨", "간단한 안내 필요", "기술적 설명 필요"]},
}

댓글 = "영상 잘 봤습니다. 그런데 유튜브 API 키 넣는 부분에서 계속 에러가 나요. 어떻게 해야 하나요?"

결과 = systemone(state=댓글, questions=질문지)
print(json.dumps(결과, ensure_ascii=False, indent=1))


{
 "댓글유형": {
  "type": "choice",
  "choice": "질문",
  "confidence": 0.801,
  "probabilities": {
   "감사": 0.139,
   "질문": 0.801,
   "스팸": 0.062
  }
 },
 "답변필요": {
  "type": "noul",
  "noul": 0.547
 },
 "난이도": {
  "type": "score",
  "score": 1.3,
  "probabilities": {
   "0": 0.074,
   "1": 0.547,
   "2": 0.377
  }
 }
}


### 코드는 이렇게 움직입니다 (branch · route)

확신도가 낮으면 행동하지 않고 사람을 부릅니다. 이게 자동화 사고를 막는 안전장치입니다.


In [ ]:
답 = 결과["댓글유형"]
if 답["confidence"] < 0.6:
    print("→ 애매함: 사람(운영자)에게 넘김")
elif 답["choice"] == "질문" and 결과["답변필요"]["noul"] > 0.5:
    print("→ 질문으로 확신: 답변 초안 작성 대기열로")
elif 답["choice"] == "스팸":
    print("→ 스팸: 숨김 처리 후보")
else:
    print("→ 감사 댓글: 짧은 화답 초안")


→ 질문으로 확신: 답변 초안 작성 대기열로


### 과제: 여러분의 질문지를 설계하세요

1. `댓글` 대신 여러분 업무의 문장을 넣어보세요 (고객 문의, 리뷰, 이메일…)
2. `질문지`의 질문과 보기를 여러분 업무에 맞게 바꿔보세요 — **이 설계가 자동화의 품질을 정합니다**
3. 보기 순서를 바꾸거나 "기타" 보기를 추가해서 확률이 어떻게 변하는지 관찰해보세요

기억할 것: 질문과 보기는 언제나 사람이 정합니다. 모델은 만들지 않고, 나누기만 합니다.
